# CySent Qwen GPU Evaluation

## 0. Purpose and scientific constraints

Google Colab is an evaluation compute environment, not part of the CySent architecture.

Hugging Face remains the model artifact source. This notebook uses the repository's real `HFAgent`, `AgentRouter`, environment, action mapping, and benchmark runner. It does not reimplement research semantics.

The default path evaluates the requested merged artifact on CUDA without quantization. It never substitutes a base model, adapter, mock model, or PPO action for Qwen. Smoke results are correctness evidence only, not performance claims. Full Qwen and Hybrid benchmarks remain disabled until the frozen runner gains per-episode incremental persistence and resume support.


## 1. Configuration

Edit only this cell to move between Colab, another CUDA Jupyter runtime, or a local GPU workstation.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/sxchin-01/CySent.git"
REPO_REVISION = os.environ.get("CYSENT_REPO_REVISION", "main")  # Set to the P3D commit SHA for an authoritative run.
MODEL_ID = "sxchin01/CySent-Qwen-RL-merged"
MODEL_REVISION = "fb75512b037bb37de575916afd900c03ab860cb5"  # Verified merged artifact revision.

WORKSPACE_ROOT = Path(os.environ.get("CYSENT_EVAL_WORKSPACE", Path.cwd() / "cysent_eval_workspace")).expanduser().resolve()
REPO_ROOT = WORKSPACE_ROOT / "repo"
OUTPUT_ROOT = Path(os.environ.get("CYSENT_EVAL_OUTPUT", WORKSPACE_ROOT / "research_outputs")).expanduser().resolve()
MODEL_CACHE_ROOT = WORKSPACE_ROOT / "model_cache"
PPO_MODEL_PATH = Path(os.environ.get("CYSENT_PPO_MODEL_PATH", REPO_ROOT / "backend" / "train" / "artifacts" / "best_model" / "best_model.zip")).expanduser().resolve()
EXPECTED_PPO_SHA256 = "4D4955993DD2D98CC3B8D3319C1CDA92F10EFBA95407B3D78D9763E5D968D1AF"

MAX_NEW_TOKENS = 16  # Frozen HFAgent value; changing it requires a reviewed source change.
INFERENCE_TIMEOUT_SECONDS = 600.0
BENCHMARK_SEEDS = (42, 43, 44)
BENCHMARK_MAX_STEPS = 150
BENCHMARK_CASES = (
    {"scenario": "bank", "difficulty": "hard", "attacker": "ransomware_gang"},
    {"scenario": "saas", "difficulty": "hard", "attacker": "silent_apt"},
    {"scenario": "hospital", "difficulty": "medium", "attacker": "insider_saboteur"},
)
REQUIRE_CUDA = True
ENABLE_QUANTIZATION = False  # Must remain explicit; current HFAgent merged path is unquantized.
ENABLE_BENCHMARK_EXECUTION = False
DOWNLOAD_EXPORT_IN_COLAB = False

for directory in (WORKSPACE_ROOT, OUTPUT_ROOT, MODEL_CACHE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

assert MAX_NEW_TOKENS == 16, "The frozen HFAgent uses max_new_tokens=16."
assert not ENABLE_QUANTIZATION, (
    "Quantization is not silently supported by the current HFAgent local merged-model path. "
    "Keep it off for the primary evaluation."
)
print({"workspace": str(WORKSPACE_ROOT), "outputs": str(OUTPUT_ROOT), "model_id": MODEL_ID})


## 2. Runtime preflight

This reports the existing runtime and does not install or replace PyTorch. CUDA absence is a hard failure when `REQUIRE_CUDA` is true. Colab-specific behavior is isolated in these helpers.

In [ ]:
import getpass
import importlib.util
import json
import platform
import subprocess
import sys

import torch

IS_COLAB = importlib.util.find_spec("google.colab") is not None

def get_colab_secret(name):
    if not IS_COLAB:
        return None
    from google.colab import userdata
    try:
        return userdata.get(name)
    except Exception:
        return None

def offer_colab_download(path):
    if IS_COLAB and DOWNLOAD_EXPORT_IN_COLAB:
        from google.colab import files
        files.download(str(path))

def git_output(cwd, *args):
    result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True, check=False)
    return result.stdout.strip() if result.returncode == 0 else None

runtime = {
    "python": sys.version.replace("\n", " "),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "torch_cuda": torch.version.cuda,
    "colab": IS_COLAB,
}
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    runtime.update({
        "gpu_name": torch.cuda.get_device_name(0),
        "gpu_total_gib": round(total_bytes / 1024**3, 3),
        "gpu_free_gib": round(free_bytes / 1024**3, 3),
    })
print(json.dumps(runtime, indent=2))
smi = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=False)
print(smi.stdout if smi.returncode == 0 else f"nvidia-smi unavailable: {smi.stderr.strip()}")
cwd_revision = git_output(Path.cwd(), "rev-parse", "HEAD")
cwd_status = git_output(Path.cwd(), "status", "--short") if cwd_revision else None
print({"current_directory_git_revision": cwd_revision, "dirty": bool(cwd_status), "status": cwd_status})
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this evaluation, but torch.cuda.is_available() is false.")


## 3. Repository and version verification

The requested revision is resolved and the checkout is detached at its immutable commit. Set `CYSENT_REPO_REVISION` to the P3D commit SHA for an authoritative run. Existing dirty repositories are never overwritten.

In [ ]:
def run_checked(command, cwd=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, cwd=cwd, text=True, check=True)

if not (REPO_ROOT / ".git").exists():
    run_checked(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)])
else:
    existing_status = git_output(REPO_ROOT, "status", "--short")
    if existing_status:
        raise RuntimeError(f"Refusing to alter dirty repository at {REPO_ROOT}:\n{existing_status}")

run_checked(["git", "fetch", "--depth", "1", "origin", REPO_REVISION], cwd=REPO_ROOT)
run_checked(["git", "checkout", "--detach", REPO_REVISION], cwd=REPO_ROOT)
resolved_repo_revision = git_output(REPO_ROOT, "rev-parse", "HEAD")
repo_status = git_output(REPO_ROOT, "status", "--short") or ""
if len(REPO_REVISION) == 40 and all(character in "0123456789abcdefABCDEF" for character in REPO_REVISION) and resolved_repo_revision.lower() != REPO_REVISION.lower():
    raise RuntimeError(f"Repository revision mismatch: {resolved_repo_revision} != {REPO_REVISION}")
if repo_status:
    raise RuntimeError(f"Repository is unexpectedly dirty:\n{repo_status}")
if not PPO_MODEL_PATH.is_relative_to(REPO_ROOT):
    raise RuntimeError("CYSENT_PPO_MODEL_PATH must be inside REPO_ROOT because frozen benchmark metadata records a repository-relative path.")
PPO_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
parser_source = (REPO_ROOT / "backend" / "agents" / "hf_agent.py").read_text(encoding="utf-8")
if 'completion_tokens = output[0, input_token_count:]' not in parser_source:
    raise RuntimeError("Required P3B completion-only parsing fix is absent.")
print({"repo_revision": resolved_repo_revision, "clean": True, "p3b_parser_fix": True})


## 4. Dependencies

Only backend evaluation dependencies are installed. PyTorch is deliberately omitted so the runtime's CUDA build is preserved. PEFT, TRL, datasets, OpenEnv, and frontend dependencies are not required for an unquantized merged-model evaluation.

In [ ]:
from importlib.metadata import version

torch_package_version_before = version("torch")
evaluation_requirements = [
    "gymnasium>=1.0.0",
    "numpy>=1.26.0",
    "pandas>=2.2.0",
    "matplotlib>=3.9.0",
    "stable-baselines3>=2.5.0",
    "transformers>=4.40.0",
    "accelerate>=0.30.0",
    "huggingface_hub>=0.23.0",
    "safetensors",
    "pyyaml",
]
torch_constraint = WORKSPACE_ROOT / "torch-constraint.txt"
torch_constraint.write_text(f"torch=={torch_package_version_before}\n", encoding="utf-8")
run_checked([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", "--constraint", str(torch_constraint), *evaluation_requirements])

package_names = ["torch", "gymnasium", "numpy", "pandas", "matplotlib", "stable-baselines3", "transformers", "accelerate", "huggingface_hub", "safetensors", "pyyaml"]
package_versions = {name: version(name) for name in package_names}
print(json.dumps(package_versions, indent=2))
if package_versions["torch"] != torch_package_version_before:
    raise RuntimeError("Dependency installation changed PyTorch; restart from a clean runtime and stop.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA disappeared after dependency installation; stop rather than replacing PyTorch.")


## 5. Secure Hugging Face authentication

Authentication order is existing `HF_TOKEN`, Colab secret `HF_TOKEN`, then hidden interactive input. The token is never printed or written to research metadata.

In [ ]:
hf_token = os.environ.get("HF_TOKEN", "").strip()
token_source = "environment" if hf_token else None
if not hf_token:
    hf_token = (get_colab_secret("HF_TOKEN") or "").strip()
    token_source = "colab_secret" if hf_token else None
if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (input hidden): " ).strip()
    token_source = "interactive" if hf_token else None
if not hf_token:
    raise RuntimeError("A Hugging Face token is required to resolve the requested artifact.")
os.environ["HF_TOKEN"] = hf_token
print({"authenticated": True, "token_source": token_source})


## 6. Model artifact identity

The requested symbolic revision is resolved before model download. Claims produced from this notebook identify the resolved public artifact SHA and do not assert cryptographic identity with the historical 50-episode run. Metadata is written before expensive work.

In [ ]:
import hashlib
import tempfile
from datetime import datetime, timezone

from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoConfig
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from backend.agents.qwen_rl_policy import QWEN_RL_POLICY_CONTRACT

def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False, suffix=".tmp") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")
        temporary = Path(handle.name)
    temporary.replace(path)

model_info = HfApi(token=hf_token).model_info(MODEL_ID, revision=MODEL_REVISION, files_metadata=True)
resolved_model_revision = model_info.sha
if resolved_model_revision != MODEL_REVISION:
    raise RuntimeError(f"Merged model revision mismatch: {resolved_model_revision} != {MODEL_REVISION}")
config_path = hf_hub_download(MODEL_ID, "config.json", revision=resolved_model_revision, token=hf_token, cache_dir=MODEL_CACHE_ROOT)
model_config = AutoConfig.from_pretrained(str(Path(config_path).parent), trust_remote_code=True)
model_identity = {
    "requested_model_id": MODEL_ID,
    "requested_model_revision": MODEL_REVISION,
    "resolved_model_revision": resolved_model_revision,
    "model_type": getattr(model_config, "model_type", None),
    "architectures": getattr(model_config, "architectures", None),
    "config_name_or_path": getattr(model_config, "_name_or_path", None),
}
research_metadata = {
    "schema_version": 2,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repo_url": REPO_URL,
    "repo_revision": resolved_repo_revision,
    "repo_dirty": False,
    "model": model_identity,
    "qwen_policy_contract": QWEN_RL_POLICY_CONTRACT,
    "prior_generative_smoke_diagnostic": {
        "status": "rejected_no_canonical_action",
        "completion": "The chosen action should be the most effective in the current state of the network.",
        "benchmark_data": False,
    },
    "runtime": runtime,
    "packages": package_versions,
    "quantization_enabled": ENABLE_QUANTIZATION,
    "benchmark_seeds": list(BENCHMARK_SEEDS),
    "benchmark_max_steps": BENCHMARK_MAX_STEPS,
    "benchmark_cases": list(BENCHMARK_CASES),
    "historical_run_identity_claimed": False,
    "qwen_smoke_gate_passed": False,
    "hybrid_smoke_gate_passed": False,
    "benchmark_executed": False,
}
METADATA_PATH = OUTPUT_ROOT / "metadata.json"
atomic_write_json(METADATA_PATH, research_metadata)
print(json.dumps(model_identity, indent=2))
print(f"Early metadata written to {METADATA_PATH}")


## 7. Model acquisition and local HFAgent loading

`HF_MERGED_MODEL_ID` would select hosted `InferenceClient`; it does not mean local GPU inference. The truthful local path is: resolve and download the exact Hub snapshot, clear hosted precedence variables, and pass that local merged-model directory as `adapter_path` to the real `HFAgent`. `_is_merged_model_dir` then selects `AutoModelForCausalLM`.

No quantized retry occurs. An out-of-memory error records CUDA memory evidence and stops.

In [ ]:
from huggingface_hub import snapshot_download

if ENABLE_QUANTIZATION:
    raise RuntimeError("Optional quantization requires a separately reviewed HFAgent implementation and is intentionally not activated here.")

model_snapshot = Path(snapshot_download(
    repo_id=MODEL_ID,
    revision=resolved_model_revision,
    token=hf_token,
    cache_dir=MODEL_CACHE_ROOT,
)).resolve()
if not (model_snapshot / "config.json").exists():
    raise RuntimeError("Resolved model snapshot has no config.json.")
weight_files = sorted(model_snapshot.glob("*.safetensors")) + sorted(model_snapshot.glob("pytorch_model*.bin"))
if not weight_files:
    raise RuntimeError("Resolved model snapshot contains no model weights; no substitute will be used.")

os.environ.pop("HF_ENDPOINT_URL", None)
os.environ.pop("HF_MERGED_MODEL_ID", None)
os.environ["HF_MODEL_ID"] = MODEL_ID
os.environ["HF_ADAPTER_PATH"] = str(model_snapshot)
os.environ["HF_TIMEOUT"] = str(INFERENCE_TIMEOUT_SECONDS)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from backend.agents.hf_agent import _is_merged_model_dir
from backend.agents.qwen_rl_policy import QWEN_RL_POLICY_CONTRACT, QWEN_RL_SOURCE_ID, QwenRLPolicyAgent, build_historical_rl_prompt
research_metadata["qwen_policy_contract"] = QWEN_RL_POLICY_CONTRACT
atomic_write_json(METADATA_PATH, research_metadata)

if not _is_merged_model_dir(str(model_snapshot)):
    raise RuntimeError("The resolved artifact is not classified as a merged model by the real HFAgent.")

qwen_agent = QwenRLPolicyAgent(
    model_id=MODEL_ID,
    adapter_path=str(model_snapshot),
    endpoint_url=None,
    token=hf_token,
    timeout=INFERENCE_TIMEOUT_SECONDS,
)
if qwen_agent._active_backend != "local_deferred":
    raise RuntimeError(f"Expected local_deferred backend, got {qwen_agent._active_backend!r}.")

memory_before = torch.cuda.mem_get_info()
try:
    qwen_agent._initialize_local_model()
    qwen_agent._active_backend = "local"
except torch.cuda.OutOfMemoryError as exc:
    free_after, total_after = torch.cuda.mem_get_info()
    research_metadata["model_load_failure"] = {
        "type": type(exc).__name__,
        "gpu_free_gib_before": memory_before[0] / 1024**3,
        "gpu_free_gib_after": free_after / 1024**3,
        "gpu_total_gib": total_after / 1024**3,
    }
    atomic_write_json(METADATA_PATH, research_metadata)
    torch.cuda.empty_cache()
    raise RuntimeError("Unquantized merged model did not fit. Quantization was not activated automatically.") from exc

parameter_dtypes = sorted({str(parameter.dtype) for parameter in qwen_agent.model.parameters()})
raw_device_map = getattr(qwen_agent.model, "hf_device_map", None)
device_map = ({str(key): str(value) for key, value in raw_device_map.items()} if isinstance(raw_device_map, dict) else raw_device_map)
load_identity = {
    "snapshot_path": str(model_snapshot),
    "backend": qwen_agent._active_backend,
    "using_local_model": qwen_agent._using_local_model,
    "parameter_dtypes": parameter_dtypes,
    "device_map": device_map,
    "is_quantized": bool(getattr(qwen_agent.model, "is_quantized", False)),
    "weight_files": [path.name for path in weight_files],
}
if load_identity["is_quantized"]:
    raise RuntimeError("Model unexpectedly loaded quantized while ENABLE_QUANTIZATION is false.")
research_metadata["model_load"] = load_identity
atomic_write_json(METADATA_PATH, research_metadata)
print(json.dumps(load_identity, indent=2, default=str))


## 8. Real CySent Qwen smoke gate

Three representative frozen cases use the real environment, historical RL prompt, final-token constrained logits, and seeded categorical sampling with the local merged model. Each environment advances exactly once. The earlier rejected generative sentence remains diagnostic evidence only and is not benchmark data.

In [ ]:
from backend.env.security_env import ACTION_NAMES, CySentSecurityEnv

expected_collision = {"token_id": 3400, "actions": ["patch_hr_systems", "patch_web_server", "patch_auth_server"]}
if expected_collision not in QWEN_RL_POLICY_CONTRACT["collision_groups"]:
    raise AssertionError("Historical first-token collision metadata is missing.")
qwen_smoke_rows = []
for case in BENCHMARK_CASES:
    env = CySentSecurityEnv(max_steps=BENCHMARK_MAX_STEPS, seed=42)
    observation, info = env.reset(seed=42, options={**case, "strategy_mode": "balanced", "action_source": "qwen_rl", "intelligence_enabled": False})
    step_before = env.current_step
    expected_prompt = build_historical_rl_prompt(info)
    qwen_agent.reset(42)
    try:
        action = int(qwen_agent.predict_action(info))
        diagnostics = dict(qwen_agent.last_decision)
        qwen_agent.reset(42)
        repeated_action = int(qwen_agent.predict_action(info))
    except Exception as exc:
        if env.current_step != step_before:
            raise AssertionError("Environment advanced after failed Qwen prediction.") from exc
        raise
    if action not in ACTION_NAMES:
        raise AssertionError(f"Qwen returned non-canonical action {action}.")
    if repeated_action != action:
        raise AssertionError("Seeded Qwen RL smoke action was not reproducible after reset.")
    if diagnostics["policy_mode"] != QWEN_RL_POLICY_CONTRACT["mode"] or diagnostics["source"] != QWEN_RL_SOURCE_ID:
        raise AssertionError("Qwen smoke did not use the historical constrained policy identity.")
    if len(diagnostics["constrained_logits"]) != 12 or len(diagnostics["constrained_probabilities"]) != 12:
        raise AssertionError("Qwen smoke did not expose the 12-action constrained distribution.")
    if diagnostics["action_token_ids"] != QWEN_RL_POLICY_CONTRACT["action_token_ids"]:
        raise AssertionError("Qwen smoke action-token IDs do not match the historical contract.")
    if abs(sum(diagnostics["constrained_probabilities"]) - 1.0) > 1e-5:
        raise AssertionError("Qwen constrained probabilities are not normalized.")
    if diagnostics["prompt_sha256"] != hashlib.sha256(expected_prompt.encode("utf-8")).hexdigest():
        raise AssertionError("Qwen smoke did not score the historical RL prompt.")
    if not qwen_agent._using_local_model or qwen_agent.client is not None:
        raise AssertionError("Qwen smoke did not use the local HFAgent model path.")
    observation, reward, terminated, truncated, next_info = env.step(action)
    if env.current_step != step_before + 1:
        raise AssertionError("Environment did not advance exactly once after a valid action.")
    qwen_smoke_rows.append({
        **case,
        "seed": 42,
        "source": QWEN_RL_SOURCE_ID,
        "backend": "local",
        "selected_action_id": action,
        "selected_action_name": ACTION_NAMES[action],
        "repeated_seeded_action_id": repeated_action,
        "diagnostics": diagnostics,
        "executed_action_name": next_info["last_action"],
        "step_before": step_before,
        "step_after": env.current_step,
        "valid": True,
    })

QWEN_SMOKE_GATE_PASSED = len(qwen_smoke_rows) == len(BENCHMARK_CASES) and all(row["valid"] for row in qwen_smoke_rows)
if not QWEN_SMOKE_GATE_PASSED:
    raise RuntimeError("Qwen smoke gate failed; benchmark permission is denied.")
research_metadata["qwen_smoke_gate_passed"] = True
research_metadata["qwen_smoke"] = qwen_smoke_rows
atomic_write_json(METADATA_PATH, research_metadata)
print(json.dumps(qwen_smoke_rows, indent=2))


## 9. Hybrid smoke gate

Hybrid validation requires the frozen historical PPO checkpoint. Upload or mount it at `PPO_MODEL_PATH`; its SHA-256 must match the configured historical hash. The real router is used with the already loaded real Qwen agent. Failure fallback is injected only to verify metadata and is never counted as a genuine Qwen decision.

In [ ]:
from backend.agents.identities import HISTORICAL_PPO_AGENT
from backend.agents.ppo_agent import PPOAgent
from backend.agents.router import AgentRouter

hybrid_memory_diagnostics = []
def record_hybrid_memory(label):
    snapshot = {"label": label, "utc": datetime.now(timezone.utc).isoformat()}
    try:
        import psutil
        snapshot["process_rss_bytes"] = psutil.Process().memory_info().rss
    except Exception as exc:
        snapshot["process_rss_bytes"] = None
        snapshot["psutil_status"] = f"unavailable: {type(exc).__name__}"
    if torch.cuda.is_available():
        cuda_free, cuda_total = torch.cuda.mem_get_info()
        snapshot.update({
            "cuda_allocated_bytes": torch.cuda.memory_allocated(),
            "cuda_reserved_bytes": torch.cuda.memory_reserved(),
            "cuda_free_bytes": cuda_free,
            "cuda_total_bytes": cuda_total,
        })
    hybrid_memory_diagnostics.append(snapshot)
    research_metadata["hybrid_memory_diagnostics"] = hybrid_memory_diagnostics
    atomic_write_json(METADATA_PATH, research_metadata)
    print(json.dumps(snapshot, indent=2))
    return snapshot

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()

if not PPO_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Hybrid smoke requires the historical PPO checkpoint at {PPO_MODEL_PATH}. "
        "Upload or mount it without changing PPO semantics, then rerun this cell."
    )
ppo_hash = sha256_file(PPO_MODEL_PATH)
if ppo_hash != EXPECTED_PPO_SHA256:
    raise RuntimeError(f"Historical PPO hash mismatch: {ppo_hash}")

record_hybrid_memory("before_ppo_load")
historical_ppo_agent = PPOAgent(str(PPO_MODEL_PATH), artifact_id="historical_ppo")
record_hybrid_memory("after_ppo_load")
record_hybrid_memory("before_router_construction")
router = AgentRouter(
    config={"default_agent": "ppo_agent", "mode": "hybrid", "hybrid_threshold": 10, "full_llm": False, "hf_policy_mode": "qwen_rl"},
    ppo_agent=historical_ppo_agent,
    hf_agent=qwen_agent,
)
record_hybrid_memory("after_router_construction")
if router.ppo_agent is not historical_ppo_agent or router.hf_agent is not qwen_agent:
    raise AssertionError("Hybrid router did not retain the injected PPO and Qwen agent objects.")
router.switch_agent("hybrid")

env = CySentSecurityEnv(max_steps=BENCHMARK_MAX_STEPS, seed=42)
observation, info = env.reset(seed=42, options={**BENCHMARK_CASES[0], "intelligence_enabled": False})
low_state = dict(info)
low_state["network_risk"] = min(float(low_state["network_risk"]), 0.2)

router.reset_episode(42)
low_action = int(router.predict_action(observation, low_state))
low_source = router.last_used_agent
if low_source != HISTORICAL_PPO_AGENT:
    raise AssertionError(f"Expected low-risk PPO source, got {low_source}.")
for _ in range(8):
    router.predict_action(observation, low_state)
record_hybrid_memory("before_periodic_qwen_decision")
periodic_action = int(router.predict_action(observation, low_state))
record_hybrid_memory("after_periodic_qwen_decision")
periodic_source = router.last_used_agent
if periodic_source != QWEN_RL_SOURCE_ID:
    raise AssertionError(f"Expected periodic Qwen source on turn 10, got {periodic_source}.")

router.reset_episode(42)
high_state = dict(low_state)
high_state["network_risk"] = 0.8
record_hybrid_memory("before_high_risk_qwen_decision")
high_action = int(router.predict_action(observation, high_state))
record_hybrid_memory("after_high_risk_qwen_decision")
high_source = router.last_used_agent
if high_source != QWEN_RL_SOURCE_ID:
    raise AssertionError(f"Expected high-risk Qwen source, got {high_source}.")

class _InjectedFailingHF:
    def is_available(self):
        return True
    def predict_action(self, state):
        raise RuntimeError("injected smoke failure")

router.reset_episode(42)
router.hf_agent = _InjectedFailingHF()
fallback_action = int(router.predict_action(observation, high_state))
fallback_source = router.last_used_agent
fallback_reason = router.last_fallback_reason
router.hf_agent = qwen_agent
if fallback_source != HISTORICAL_PPO_AGENT or not fallback_reason:
    raise AssertionError("Injected Qwen failure was not observably attributed to PPO fallback.")

hybrid_smoke = {
    "ppo_sha256": ppo_hash,
    "injected_agent_identity_verified": True,
    "memory_diagnostics": hybrid_memory_diagnostics,
    "low_risk": {"action": low_action, "source": low_source},
    "periodic_turn_10": {"action": periodic_action, "source": periodic_source},
    "high_risk": {"action": high_action, "source": high_source},
    "injected_failure": {"action": fallback_action, "source": fallback_source, "fallback_reason": fallback_reason, "genuine_qwen_decision": False},
}
HYBRID_SMOKE_GATE_PASSED = True
research_metadata["hybrid_smoke_gate_passed"] = True
research_metadata["hybrid_smoke"] = hybrid_smoke
atomic_write_json(METADATA_PATH, research_metadata)
print(json.dumps(hybrid_smoke, indent=2))


## 10. Controlled benchmark

The frozen matrix is already implemented by `backend/train/benchmark_agents.py`, including requested actions, executed actions, underlying sources, fallback count, latency, reward, breach, risk, uptime, cost, survival, substitutions, and failures.

The enhanced runner writes metadata before execution, atomically persists each completed episode or failure, and resumes by `(agent, case_id, seed)` only when its configuration, model revision, checkpoint, commit, and source hashes still match. Qwen and Hybrid run serially through the real CySent agents.

In [ ]:
if not globals().get("QWEN_SMOKE_GATE_PASSED", False):
    raise RuntimeError("Qwen smoke gate has not passed.")
if not globals().get("HYBRID_SMOKE_GATE_PASSED", False):
    raise RuntimeError("Hybrid smoke gate has not passed.")

BENCHMARK_RUNNER_SUPPORTS_INCREMENTAL_RESUME = True
RUNNER_MODEL_METADATA_PATH = OUTPUT_ROOT / "runner_model_metadata.json"
runner_model_metadata = {
    "model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION,
    "resolved_revision": resolved_model_revision,
    "dtype": parameter_dtypes,
    "quantization": load_identity["is_quantized"],
    "device": device_map,
    "gpu_name": runtime.get("gpu_name"),
    "cuda_version": runtime.get("torch_cuda"),
    "python_version": runtime.get("python"),
    "torch_version": package_versions.get("torch"),
    "numpy_version": package_versions.get("numpy"),
    "transformers_version": package_versions.get("transformers"),
    "accelerate_version": package_versions.get("accelerate"),
    "huggingface_hub_version": package_versions.get("huggingface_hub"),
    "stable_baselines3_version": package_versions.get("stable-baselines3"),
}
atomic_write_json(RUNNER_MODEL_METADATA_PATH, runner_model_metadata)
benchmark_environment = os.environ.copy()
benchmark_environment.update({
    "HF_ENDPOINT_URL": "",
    "HF_MERGED_MODEL_ID": "",
    "HF_MODEL_ID": MODEL_ID,
    "HF_ADAPTER_PATH": str(model_snapshot),
    "HF_TIMEOUT": str(INFERENCE_TIMEOUT_SECONDS),
})

def benchmark_command(agent_name):
    outdir = OUTPUT_ROOT / "benchmarks" / f"p3_{agent_name}"
    return [
        sys.executable, "-m", "backend.train.benchmark_agents",
        "--agents", agent_name,
        "--seeds", ",".join(str(seed) for seed in BENCHMARK_SEEDS),
        "--max-steps", str(BENCHMARK_MAX_STEPS),
        "--ppo-path", str(PPO_MODEL_PATH),
        "--outdir", str(outdir),
        "--resume",
        "--model-metadata", str(RUNNER_MODEL_METADATA_PATH),
    ]

prepared_commands = {
    "qwen": benchmark_command("qwen_rl"),
    "hybrid": benchmark_command("hybrid_router"),
}
print(json.dumps(prepared_commands, indent=2))
if ENABLE_BENCHMARK_EXECUTION:
    if not BENCHMARK_RUNNER_SUPPORTS_INCREMENTAL_RESUME:
        raise RuntimeError("Benchmark blocked: frozen runner lacks per-episode persistence and resume support.")
    import gc
    for name in ("router", "qwen_agent"):
        if name in globals():
            del globals()[name]
    gc.collect()
    torch.cuda.empty_cache()
    for command in prepared_commands.values():
        subprocess.run(command, cwd=REPO_ROOT, env=benchmark_environment, check=True)
else:
    print("Benchmark execution is disabled. No performance episodes were run.")


## 11. Results validation

Run this only after an incremental/resumable backend runner has generated results. It derives source counts from the preserved per-decision `underlying_agents` field and rejects Qwen or Hybrid evidence with no genuine Qwen decisions.

In [ ]:
import csv
from collections import Counter

def validate_benchmark_directory(directory, requested_agent, expected_episodes=9):
    directory = Path(directory)
    episode_path = directory / "episodes.csv"
    failure_path = directory / "failures.csv"
    metadata_path = directory / "metadata.json"
    if not all(path.exists() for path in (episode_path, failure_path, metadata_path)):
        raise FileNotFoundError(f"Incomplete benchmark directory: {directory}")
    with episode_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != expected_episodes:
        raise AssertionError(f"Expected {expected_episodes} completed episodes, found {len(rows)}.")
    if any(row["agent"] != requested_agent for row in rows):
        raise AssertionError("Requested agent identity mismatch in episode rows.")
    source_counts = Counter()
    for row in rows:
        source_counts.update(json.loads(row["underlying_agents"]))
    with failure_path.open(newline="", encoding="utf-8") as handle:
        failures = list(csv.DictReader(handle))
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("status") != "complete":
        raise AssertionError(f"Benchmark metadata is not complete: {metadata.get('status')!r}.")
    if metadata.get("qwen_model", {}).get("resolved_revision") != resolved_model_revision:
        raise AssertionError("Benchmark model revision does not match the smoke-tested model.")
    result = {
        "requested_agent": requested_agent,
        "completed_episodes": len(rows),
        "qwen_decisions": source_counts[QWEN_RL_SOURCE_ID],
        "ppo_decisions": source_counts[HISTORICAL_PPO_AGENT],
        "fallback_count": sum(int(row["fallback_count"]) for row in rows),
        "failure_count": len(failures),
    }
    if result["qwen_decisions"] == 0:
        raise AssertionError("No genuine Qwen decisions; this cannot be reported as Qwen/Hybrid performance evidence.")
    return result

print("Validation helper ready; no benchmark result is claimed by this notebook.")


## 12. Export

This packages current research outputs without committing them. Google Drive is optional; set `OUTPUT_ROOT` to a mounted persistent location manually if desired.

In [ ]:
import shutil

research_metadata["exported_utc"] = datetime.now(timezone.utc).isoformat()
atomic_write_json(METADATA_PATH, research_metadata)
archive_base = WORKSPACE_ROOT / f"cysent_qwen_evaluation_{resolved_model_revision[:12]}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_ROOT))
print({"archive": str(archive_path), "bytes": archive_path.stat().st_size})
offer_colab_download(archive_path)


## 13. Shutdown and cleanup notes

1. Export outputs before ending an ephemeral session.
2. Do not present smoke rows as performance evidence.
3. Do not report Hybrid performance unless genuine Qwen source counts are nonzero.
4. Do not delete the model cache until outputs are packaged if artifact inspection is still needed.
5. The following cell releases in-process model references and CUDA cache; it does not delete artifacts or outputs.

In [ ]:
import gc

for name in ("router", "qwen_agent"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("In-process model references released. Files and research outputs were preserved.")
